In [42]:
import numpy as np 
import cv2 
from plotly import express as px 
import msicpe 
import msicpe.tsi as tsi 
print(msicpe.__version__)

1.0.22


## 1 Filtrage dans l’espace direct : détection de contours

In [43]:
im1 = cv2.imread('flower.png',0)
h, w = im1.shape
print(f"Dimensions de l'image : {h} x {w}")

fig = px.imshow(im1, title="Flower", color_continuous_scale='gray') 
fig.show()

Dimensions de l'image : 256 x 256


In [48]:
Hh = np.array([[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]])

Hv = np.array([[-1, -2, -1],
               [ 0,  0,  0],
               [ 1,  2,  1]])

Gh = cv2.filter2D(im1, -1, Hh)
Gv = cv2.filter2D(im1, -1, Hv)

Gh_norm = Gh / np.max(Gh)
Gv_norm = Gv / np.max(Gv)

G = np.sqrt(Gh_norm**2 + Gv_norm**2)
G_norm = G / np.max(G)

In [49]:
fig = px.imshow(np.array([G_norm, Gh_norm, Gv_norm]), title='Affichage de composantes du grandient et sa norme', color_continuous_scale='gray', facet_col_wrap=3, facet_col=0 ) # nb d'images par ligne 
tsi.add_legend(fig,['G','Gh', 'Gv']) 
fig.show()

In [51]:
im_gauss = tsi.add_gaussian_noise(im1,50)
fig = px.imshow(im_gauss, title="Flower bruit blanc gaussien centré", color_continuous_scale='gray') 
fig.show()

In [52]:
Gh_gauss = cv2.filter2D(im_gauss, -1, Hh)
Gv_gauss  = cv2.filter2D(im_gauss, -1, Hv)

Gh_gauss_norm = Gh_gauss / np.max(Gh_gauss)
Gv__gauss_norm = Gv_gauss / np.max(Gv_gauss)

G_gauss = np.sqrt(Gh_gauss_norm**2 + Gv__gauss_norm**2)
G_gauss_norm = G_gauss / np.max(G_gauss)

fig = px.imshow(np.array([G_gauss_norm, Gh_gauss_norm, Gv__gauss_norm]), title='Affichage de composantes du grandient et sa norme en bruit blanc gaussien centré', color_continuous_scale='gray', facet_col_wrap=3, facet_col=0 ) # nb d'images par ligne 
tsi.add_legend(fig,['G_gauss','Gh_gauss', 'Gv_gauss']) 
fig.show()

## 2 Filtrage spectral : suppression de l’effet de trame

In [54]:
im = cv2.imread('journal.png',0)/255.
h, w = im2.shape
print(f"Dimensions de l'image : {h} x {w}")

fig = px.imshow(im2, title="Journal", color_continuous_scale='gray') 
fig.show()

Dimensions de l'image : 371 x 400


In [57]:
im_fft = np.fft.fftshift(np.fft.fft2(im2))
spectre = np.abs(im_fft)

fig = px.imshow(np.log(1 + spectre), title="Spectre de Fourier (log)", color_continuous_scale='gray')
fig.show()

In [58]:
lx = np.linspace(-w/2 + 0.5, w/2 - 0.5, w)
ly = np.linspace(-h/2 + 0.5, h/2 - 0.5, h)
u, v = np.meshgrid(lx, ly)

d = np.sqrt(u**2 + v**2)

p = 2
nc = 100
H = 1 / (1 + (d / nc)**(2 * p))

px.imshow(H, title="Masque du filtre de Butterworth", color_continuous_scale='gray').show()

In [ ]:
im_fft_filt = im_fft * H

spectre_filt = np.abs(im_fft_filt)
px.imshow(np.log(1 + spectre_filt), title="Spectre après filtrage", color_continuous_scale='gray').show()

im_filt = np.real(np.fft.ifft2(np.fft.ifftshift(im_fft_filt)))

px.imshow(im_filt, title="Image débruitée (sans trame)", color_continuous_scale='gray').show()